# Setup 

In [ ]:
#These packages with version numbers are need to be installed

need_package <- function(pkg,version)
{
    need <- FALSE
    if (!require(pkg,character.only = TRUE))
    {
        need <- TRUE
    } else {
        if (packageVersion(pkg) != version)
        {
            need <- TRUE
        }
    }
    return(need)
}

using_version <- function(pkg,version)
{
   if (need_package(pkg,version))
   {
        devtools::install_version(pkg, version = version,upgrade=FALSE)
   }
}

read_bucket <- function (export_path)
{
    col_types <- NULL
    bind_rows(map(system2("gsutil", args = c("ls", export_path),
        stdout = TRUE, stderr = TRUE), function(csv) {
        message(str_glue("Loading {csv}."))
        chunk <- read_csv(pipe(str_glue("gsutil cat {csv}")),
            col_types = col_types, show_col_types = FALSE)
        if (is.null(col_types)) {
            col_types <- spec(chunk)
        }
        chunk
    }))
}

download_data <- function (query)
{
    tb <- bq_project_query(Sys.getenv("GOOGLE_PROJECT"), query)
    bq_table_download(tb, page_size = 1e+05)
}
using_version("Hmisc","4.8-0")
using_version("rms","6.4-1")
using_version("ggsci","3.0.0")
using_version("miceadds","3.16-18")
using_version("ggpubr","0.5.0")
#load packages and dataset 

library(stringr)
library(bigrquery)
library(data.table)
library(readr)
library(Hmisc)
library(ggplot2)
library(ggrepel)
library(viridis)
library(ggsci)
#library(miceadds)
library(rms)
library(ggpubr)
library(tidyverse)
library(lubridate)
library(dplyr)
library(tidyr)
library(jsonlite)
library(nanoparquet)
library(furrr)     # for parallel mapping
library(purrr)
library(speedglm)
library(progressr)

dataset <- Sys.getenv("WORKSPACE_CDR")
my_bucket <- Sys.getenv("WORKSPACE_BUCKET")
load_stored_result <- TRUE

In [ ]:
#install.packages("furrr")

In [ ]:
#install.packages(c( "progressr"))

In [ ]:
#install.packages("purrr")

In [ ]:
#install.packages("speedglm")

In [ ]:
install.packages("vioplot")

# PheWas Functions 

In [ ]:
source("phewas_functions.R")

# Load Data

In [ ]:
library(dplyr)

# Read the file using read.table
pgrs_average <- read.table("PGRS_Average/plinkfiles/PGRS/PGRS.txt",
                           header = TRUE,
                           stringsAsFactors = FALSE,
                           check.names = FALSE)

# Convert to tibble and rename columns
pgrs_average <- as_tibble(pgrs_average) %>%
  rename(person_id = IID,
         score = SCORESUM)


In [ ]:
ancestry <- read_tsv("./PGRS_Average/plinkfiles/filter/ancestry_preds.tsv")


In [ ]:
dim(ancestry)

In [ ]:
pca_features_df <- ancestry %>%
  # Rename research_id to participant_id
  rename(participant_id = research_id) %>%
  # Convert the pca_features string to a list column
  mutate(pca_features = map(pca_features, ~fromJSON(.x))) %>%
  # Unnest the pca_features list column
  unnest_wider(pca_features, names_sep = "_") %>%
  # Select participant_id and the first 10 PCA features
  dplyr::select(participant_id, starts_with("pca_features_") & ends_with(as.character(0:9)))

# Rename the PCA feature columns to be more readable
colnames(pca_features_df) <- c("person_id", paste0("pca_", 1:17))


In [ ]:
head(ancestry)

In [ ]:
final_dat <- read_parquet("final_data_v8_5_30_no_daily_one_day_less_strict_hours.parquet") #read.csv("./plinkfiles/final_data/final_dat_pheWasAverage_no_day_sex_res.csv")

In [ ]:
head(final_dat)

In [ ]:
patient_list <- read_tsv("PGRS_Average/plinkfiles/filter/patient_list.txt")

# Merge final_dat and pgrs_average

In [ ]:
# Assuming final_dat already exists and has a person_id column

# First, let's create a minimal dataframe from pgrs_average with just person_id and score
pgrs_score <- pgrs_average %>% 
  dplyr::select(person_id, score)

# Now, let's join this with final_dat
final_dat <- final_dat %>%
  left_join(pgrs_score, by = "person_id")



In [ ]:
# Ensure that person_id is of the same type in both dataframes
pca_features_df <- pca_features_df %>%
  mutate(person_id = as.numeric(person_id))

final_dat <- final_dat %>%
  mutate(person_id = as.numeric(person_id))

# Merge the dataframes
final_dat <- final_dat %>%
  left_join(pca_features_df, by = "person_id")

# Drop all rows with NA in Score
This would correspond to ancestory thats not EUR or patients that do not have genetic data available. 

Only drop rows where relevant
* `pca_1` -> Genetic data 
* `sex_concenpt` -> no sex infromation
* `data_of_birth` -> no age 

All covariants in PheWas model

In [ ]:
# Assuming 'final_dat' is your data frame

# Store the original number of rows
original_rows <- nrow(final_dat)

# Define the columns to check for NAs
cols_to_check_for_na <- c("sex_concept", "pca_1", "date_of_birth","score")

# Verify that the columns exist in the data frame
cols_to_check_for_na <- intersect(cols_to_check_for_na, colnames(final_dat))

# Print the columns that will be checked for NAs
cat("Columns where NAs will lead to row removal:\n")
if (length(cols_to_check_for_na) > 0) {
  print(cols_to_check_for_na)
} else {
  cat("None of the specified columns exist in the data frame.\n")
}
cat("\n")

# Initialize a data frame to store the removed rows
removed_dat <- tibble()

if (length(cols_to_check_for_na) > 0) {
  # Identify the rows that HAVE an NA in ANY of the specified columns
  rows_to_remove_full <- final_dat %>%
    filter(if_any(all_of(cols_to_check_for_na), is.na))

  if (nrow(rows_to_remove_full) > 0) {
    # Create the 'removed_dat' data frame
    removed_dat <- rows_to_remove_full %>%
      dplyr::select(all_of(cols_to_check_for_na))

    # Calculate NA counts per column in removed_dat
    na_counts <- removed_dat %>%
      summarise(across(everything(), ~sum(is.na(.)))) %>%
      pivot_longer(cols = everything(), names_to = "column", values_to = "na_count")
  }

  # Drop these identified rows from the main data frame
  final_dat <- final_dat %>%
    drop_na(all_of(cols_to_check_for_na))
} else {
  cat("No specified columns found in the data frame, so no rows dropped.\n")
  na_counts <- tibble(column = character(0), na_count = integer(0)) # create empty tibble to avoid errors later
}

# --- REPORTING ---

# Calculate the number of rows dropped
rows_dropped_count <- original_rows - nrow(final_dat)

# Print the results
cat("Number of original rows:", original_rows, "\n")
cat("Number of rows dropped due to NAs in specified columns:", rows_dropped_count, "\n")
cat("Final number of rows in 'final_dat':", nrow(final_dat), "\n")
cat("Number of rows in 'removed_dat':", nrow(removed_dat), "\n")
cat("Final number of columns in 'final_dat':", ncol(final_dat), "\n\n")

# Display a sample of rows from the 'removed_dat' data frame
if (nrow(removed_dat) > 0) {
  cat("Showing a sample from 'removed_dat' (up to 5 rows):\n")
  print(head(removed_dat, 5))
  cat("\n... (Inspect the NAs in these rows.)\n")

  # Sanity check
  if (nrow(removed_dat) != rows_dropped_count) {
    cat(paste0("Note: Count of rows in 'removed_dat' (", nrow(removed_dat),
               ") does not match 'rows_dropped_count' (", rows_dropped_count, ").\n"))
  }

  # Print NA counts per column
  cat("\nNA counts per column in 'removed_dat':\n")
  print(na_counts)

} else if (rows_dropped_count > 0) {
  cat("Note: Rows were dropped from 'final_dat', but 'removed_dat' is empty.\n")
} else {
  cat("No rows were removed based on the specified NA criteria, so 'removed_dat' is empty.\n")
}

# Print the first few rows of the updated 'final_dat'
cat("\nFirst few rows of the *final* updated 'final_dat':\n")
print(head(final_dat))

### Calculate Age (Last monitoring) 

In [ ]:
final_dat <- final_dat %>%
  mutate(age = as.numeric(difftime(as.Date(max_date), as.Date(date_of_birth), units = "days")) / 365.25)

In [ ]:
# convert to hours;
final_dat$average_daily_sleep <- final_dat$average_daily_sleep / 60; 

In [ ]:
head(final_dat$score)

# Calculate Linear PGRS residual Model

In [ ]:
final_dat$sex_concept <- as.factor(final_dat$sex_concept)

final_dat$score_z <- scale(final_dat$score)

sleep.res2 <- lm(average_daily_sleep ~ score_z + age + I(age^2) + sex_concept + 
                 pca_1 + pca_2 + pca_3 + pca_4 + pca_5 + 
                 pca_6 + pca_7 + pca_8 + pca_9 + pca_10, 
                 data = final_dat)


In [ ]:
library(ggplot2)

ggplot(final_dat, aes(x = average_daily_sleep)) +
  geom_histogram(aes(y = ..density..), bins = 30, fill = "lightblue", color = "black") +
  geom_density(color = "red", linewidth = 1) +
  theme_minimal() +
  labs(
    title = "Distribution of Sleep Length",
    x = "Sleep length",
    y = "Density"
  ) +
  theme(
    plot.title = element_text(hjust = 0.5, size = 14),
    axis.title = element_text(size = 12)
  )


In [ ]:
# Step 1: Calculate residuals and add them to final_dat
final_dat <- final_dat %>% 
  mutate(sleep.dev = (residuals(sleep.res2)),
         sleep.dev_abs = abs(residuals(sleep.res2)))


### Result of model

In [ ]:
# Get the summary of the regression model
model_summary <- summary(sleep.res2)

# Extract coefficients, standard errors, t-values, and p-values
coef_table <- coef(model_summary)

# Create a data frame with the results
results <- data.frame(
  Predictor = rownames(coef_table),
  Coefficient = coef_table[, "Estimate"],
  Std_Error = coef_table[, "Std. Error"],
  t_value = coef_table[, "t value"],
  p_value = coef_table[, "Pr(>|t|)"]
)

# Print the results
(results)


In [ ]:
# Print R-squared and adjusted R-squared
cat("\nR-squared:", model_summary$r.squared)
cat("\nAdjusted R-squared:", model_summary$adj.r.squared)

# Print F-statistic and its p-value
cat("\nF-statistic:", model_summary$fstatistic[1])
cat("\nF-statistic p-value:", pf(model_summary$fstatistic[1], 
                                 model_summary$fstatistic[2], 
                                 model_summary$fstatistic[3], 
                                 lower.tail = FALSE))

In [ ]:
# Step 2: Create scatter plot
ggplot(final_dat, aes(x = sleep.dev, y = score)) + 
  geom_point(alpha = 0.5) + 
  geom_smooth(method = "lm", color = "red") + 
  theme_bw() +
  labs(x = "Sleep Time Residuals", 
       y = "Genetic Score",
       title = "Relationship between Sleep Time Residuals and Genetic Score")

# Optional: Correlation test
cor_result <- cor.test(final_dat$sleep.dev, final_dat$score, method = "pearson")
print(cor_result)


In [ ]:
# Step 2: Create scatter plot
ggplot(final_dat, aes(x = score, y = average_daily_sleep)) + 
  geom_point(alpha = 0.5) + 
  geom_smooth(method = "lm", color = "red") + 
  theme_bw() +
  labs(x = "PGRS", 
       y = "Average_Daily_sleep",
       title = "Relationship between Sleep Time  and Genetic Score")

In [ ]:
positive_sleep <- final_dat %>% filter(sleep.dev > 0)
negative_sleep <- final_dat %>% filter(sleep.dev < 0)

In [ ]:
head(positive_sleep)

In [ ]:
# import phemap 
phemap <- read.csv("./phemap.csv")

# Run PheWas Stratified

## Visulalize Distro

In [ ]:

# Calculate thresholds
low_thresh  <- quantile(final_dat$sleep.dev, probs = 0.20, na.rm = TRUE)
high_thresh <- quantile(final_dat$sleep.dev, probs = 0.80, na.rm = TRUE)

# Create ggplot violin plot
ggplot(final_dat, aes(x = "", y = sleep.dev)) +
  geom_violin(fill = "lightblue", alpha = 0.7, color = "black") +
  geom_boxplot(width = 0.1, fill = "white", alpha = 0.8) +  # Add small boxplot inside
  geom_hline(yintercept = low_thresh, color = "red", linetype = "dashed", size = 1) +
  geom_hline(yintercept = high_thresh, color = "blue", linetype = "dashed", size = 1) +
  annotate("text", x = 1.3, y = low_thresh, 
           label = paste("20th percentile:", round(low_thresh, 2)), 
           color = "red", size = 3.5, hjust = 0) +
  annotate("text", x = 1.3, y = high_thresh, 
           label = paste("80th percentile:", round(high_thresh, 2)), 
           color = "blue", size = 3.5, hjust = 0) +
  labs(title = "Distribution of sleep.dev with Percentile Cutoffs",
       subtitle = "Violin plot shows probability density, with 20th and 80th percentiles marked",
       x = "",
       y = "sleep.dev") +
  theme_minimal() +
  theme(axis.text.x = element_blank(),
        axis.ticks.x = element_blank(),
        plot.title = element_text(hjust = 0.5),
        plot.subtitle = element_text(hjust = 0.5))


In [ ]:
# 1a. Compute the 20th and 80th percentile of the residuals
low_thresh  <- quantile(final_dat$sleep.dev, probs = 0.20, na.rm = TRUE)
high_thresh <- quantile(final_dat$sleep.dev, probs = 0.80, na.rm = TRUE)

# Add Quantile variable
final_dat <- final_dat %>%
  mutate(
    resid_quantile = ntile(sleep.dev, 5),
    # Or flag directly:
    stratum = case_when(
      sleep.dev <= low_thresh  ~ "low_sleepers",
      sleep.dev >= high_thresh ~ "high_sleepers",
      TRUE                      ~ "concordant_sleepers"
    )
  )


## Concordant

In [ ]:
#Extract concordant_data
concordant_data <- final_dat[final_dat$stratum == "concordant_sleepers", ]

In [ ]:
dim(concordant_data)

In [ ]:
run <- FALSE
if(run){
results_concordant <- run_phewas(concordant_data,
                     model_equation = "sleep.dev+age+sex_concept+pca_1+pca_2+pca_3+pca_4+pca_5",
                     exposure_var = "sleep.dev",
                     impute = TRUE,
                     save_file_path = "./PGRS_Average/concordant.csv",
                     save_models = TRUE,
                     model_save_path = "./PGRS_Average/concordant_models/",
                     phemap = phemap)
    }else{
    results_concordant <- data.table::fread("./PGRS_Average/concordant.csv")
}

In [ ]:
dat <- na.omit(results_concordant[results_concordant$n_events > 20, ]
)
# kept it in. Need to look into why it was removed. 
dat <- dat[-which(dat$concept_name == "Morbid obesity")] #Removed this in publication and so doing same here
dat$label <- NA
dat$label[dat$p_value < .001] <- dat$concept_name[dat$p_value < .001]
dat$label <- do.call(rbind,strsplit(dat$label,","))[,1]
dat$siglevel_cut <- cut(dat$p_value,breaks = c(0,.00001,.0001,.001),right=FALSE,labels = FALSE)
dat$siglevel[dat$siglevel_cut == 1] <- "p < .00001"
dat$siglevel[dat$siglevel_cut == 2] <- "p < .0001"
dat$siglevel[dat$siglevel_cut == 3] <- "p < .001"
dat$siglevel = as.factor(dat$siglevel)
dat$siglevel = factor(x=dat$siglevel,levels=c("p < .00001","p < .0001", "p < .001"))
# Remove outlier results (all non-significant) 
dat_plot <- dat[dat$odds_ratio < 5,]



options(repr.plot.res=200,repr.plot.width=10,repr.plot.height=10)
ggplot(data=dat_plot, aes(x=odds_ratio, y=-log10(p_value), color=siglevel, label=label)) +
  geom_point(color='grey60') + 
  theme_bw() +
  scale_x_continuous(breaks = seq(0,5,by=.25)) +
  scale_y_continuous(breaks = seq(0,10,by=2)) +
  geom_vline(xintercept = 1) +
  scale_color_viridis(discrete=TRUE,
                      na.translate=FALSE,
                      name="Sig. Level",
                      begin = .1,end=.8) +
  geom_text_repel(max.overlaps = 25,
                  box.padding = .5,
                  force = 1.70,
                  max.iter = 500000,
                  max.time = 21) +
  ylab(bquote(~-log[10]~ 'p-value')) +
  xlab("Odds Ratio") +
  geom_hline(yintercept=-log10(.05/nrow(dat)), col="red") +
  theme(panel.grid.minor = element_blank(),
        panel.border = element_rect(),
        text = element_text(size=18),
        axis.title = element_text(size=20),
        axis.text = element_text(size=10))+
ggtitle("Concordant Sleep Stratum PGRS")



In [ ]:
df_plot <-results_concordant

# Create the -log10(p-value) column
df_plot$minus_log10_p <- -log10(df_plot$p_value)

# Handle zero p-values (replace with very small number to avoid infinite values)
df_plot$minus_log10_p[df_plot$p_value == 0] <- 5  # or use a large number


# Add color coding based on significance and effect direction
df_plot$significance <- ifelse(df_plot$p_value < 0.05/nrow(df_plot), "Bonferroni Significant",
                              ifelse(df_plot$p_value < 0.05, "Nominally Significant", "Not Significant"))

df_plot$effect_direction <- ifelse(df_plot$estimate > 0, "Positive", "Negative")

# Create color variable combining significance and direction
df_plot$color_group <- paste(df_plot$significance, df_plot$effect_direction, sep = " - ")

ggplot(df_plot, aes(x = phecode, y = minus_log10_p, color = color_group)) +
  geom_point(alpha = 0.7, size = 2) +
  geom_hline(yintercept = -log10(0.05), color = "red", linetype = "dashed", alpha = 0.7) +
  geom_hline(yintercept = -log10(0.05/nrow(df_plot)), color = "blue", linetype = "dashed", alpha = 0.7) +
  scale_color_manual(values = c(
    "Bonferroni Significant - Positive" = "darkred",
    "Bonferroni Significant - Negative" = "darkblue", 
    "Nominally Significant - Positive" = "red",
    "Nominally Significant - Negative" = "blue",
    "Not Significant - Positive" = "lightcoral",
    "Not Significant - Negative" = "lightblue"
  )) +
  labs(title = "PheWAS Manhattan Plot - Sleep Deviation Effects",
       x = "Phecode", 
       y = "-log10(p-value)",
       color = "Significance & Direction",
       subtitle = paste("Red line: p=0.05, Blue line: Bonferroni p=", 
                       round(0.05/nrow(df_plot), 6))) +
  theme_minimal() +
  theme(axis.text.x = element_text(angle = 45, hjust = 1),
        legend.position = "bottom")


In [ ]:
head(results_concordant[order(results_concordant$p_value,decreasing=F),])

## High Sleepers

In [ ]:
#Extract concordant_data
high_sleepers_data <- final_dat[final_dat$stratum == "high_sleepers", ]

In [ ]:
dim(high_sleepers_data)

In [ ]:
run <- FALSE
if(run){
results_high_sleepers <- run_phewas(high_sleepers_data,
                     model_equation = "sleep.dev+age+sex_concept+pca_1+pca_2+pca_3+pca_4+pca_5",
                     exposure_var = "sleep.dev",
                     impute = TRUE,
                     save_file_path = "./PGRS_Average/high_sleepers.csv",
                     save_models = TRUE,
                     model_save_path = "./PGRS_Average/high_sleepers_models/",
                     phemap = phemap)
    }else{
    results_high_sleepers <- data.table::fread("./PGRS_Average/high_sleepers.csv")
}

In [ ]:
dat <- na.omit(results_high_sleepers[results_high_sleepers$n_events > 20, ]
)
# kept it in. Need to look into why it was removed. 
dat <- dat[-which(dat$concept_name == "Morbid obesity")] #Removed this in publication and so doing same here
dat$label <- NA
dat$label[dat$p_value < .001] <- dat$concept_name[dat$p_value < .001]
dat$label <- do.call(rbind,strsplit(dat$label,","))[,1]
dat$siglevel_cut <- cut(dat$p_value,breaks = c(0,.00001,.0001,.001),right=FALSE,labels = FALSE)
dat$siglevel[dat$siglevel_cut == 1] <- "p < .00001"
dat$siglevel[dat$siglevel_cut == 2] <- "p < .0001"
dat$siglevel[dat$siglevel_cut == 3] <- "p < .001"
dat$siglevel = as.factor(dat$siglevel)
dat$siglevel = factor(x=dat$siglevel,levels=c("p < .00001","p < .0001", "p < .001"))
# Remove outlier results (all non-significant) 
dat_plot <- dat[dat$odds_ratio < 5,]



options(repr.plot.res=200,repr.plot.width=10,repr.plot.height=10)
ggplot(data=dat_plot, aes(x=odds_ratio, y=-log10(p_value), color=siglevel, label=label)) +
  geom_point(color='grey60') + 
  theme_bw() +
  scale_x_continuous(breaks = seq(0,5,by=.25)) +
  scale_y_continuous(breaks = seq(0,10,by=2)) +
  geom_vline(xintercept = 1) +
  scale_color_viridis(discrete=TRUE,
                      na.translate=FALSE,
                      name="Sig. Level",
                      begin = .1,end=.8) +
  geom_text_repel(max.overlaps = 25,
                  box.padding = .5,
                  force = 1.70,
                  max.iter = 500000,
                  max.time = 21) +
  ylab(bquote(~-log[10]~ 'p-value')) +
  xlab("Odds Ratio") +
  geom_hline(yintercept=-log10(.05/nrow(dat)), col="red") +
  theme(panel.grid.minor = element_blank(),
        panel.border = element_rect(),
        text = element_text(size=18),
        axis.title = element_text(size=20),
        axis.text = element_text(size=10))+
    ggtitle("High Sleep Stratum PGRS")



In [ ]:
df_plot <-results_high_sleepers

# Create the -log10(p-value) column
df_plot$minus_log10_p <- -log10(df_plot$p_value)

# Handle zero p-values (replace with very small number to avoid infinite values)
df_plot$minus_log10_p[df_plot$p_value == 0] <- 5  # or use a large number


# Add color coding based on significance and effect direction
df_plot$significance <- ifelse(df_plot$p_value < 0.05/nrow(df_plot), "Bonferroni Significant",
                              ifelse(df_plot$p_value < 0.05, "Nominally Significant", "Not Significant"))

df_plot$effect_direction <- ifelse(df_plot$estimate > 0, "Positive", "Negative")

# Create color variable combining significance and direction
df_plot$color_group <- paste(df_plot$significance, df_plot$effect_direction, sep = " - ")

ggplot(df_plot, aes(x = phecode, y = minus_log10_p, color = color_group)) +
  geom_point(alpha = 0.7, size = 2) +
  geom_hline(yintercept = -log10(0.05), color = "red", linetype = "dashed", alpha = 0.7) +
  geom_hline(yintercept = -log10(0.05/nrow(df_plot)), color = "blue", linetype = "dashed", alpha = 0.7) +
  scale_color_manual(values = c(
    "Bonferroni Significant - Positive" = "darkred",
    "Bonferroni Significant - Negative" = "darkblue", 
    "Nominally Significant - Positive" = "red",
    "Nominally Significant - Negative" = "blue",
    "Not Significant - Positive" = "lightcoral",
    "Not Significant - Negative" = "lightblue"
  )) +
  labs(title = "PheWAS Manhattan Plot - Sleep Deviation Effects",
       x = "Phecode", 
       y = "-log10(p-value)",
       color = "Significance & Direction",
       subtitle = paste("Red line: p=0.05, Blue line: Bonferroni p=", 
                       round(0.05/nrow(df_plot), 6))) +
  theme_minimal() +
  theme(axis.text.x = element_text(angle = 45, hjust = 1),
        legend.position = "bottom")


In [ ]:
head(results_high_sleepers[order(results_high_sleepers$p_value,decreasing=F),])

## Low Sleepers

In [ ]:
#Extract concordant_data
low_sleepers_data <- final_dat[final_dat$stratum == "low_sleepers", ]

In [ ]:
dim(low_sleepers_data)

In [ ]:
run <- FALSE
if(run){
results_low_sleepers <- run_phewas(low_sleepers_data,
                     model_equation = "sleep.dev+age+sex_concept+pca_1+pca_2+pca_3+pca_4+pca_5",
                     exposure_var = "sleep.dev",
                     impute = TRUE,
                     save_file_path = "./PGRS_Average/results_low_sleepers.csv",
                     save_models = TRUE,
                     model_save_path = "./PGRS_Average/results_low_sleepers_models/",
                     phemap = phemap)
    }else{
    results_low_sleepers <- data.table::fread("./PGRS_Average/results_low_sleepers.csv")
}

In [ ]:
dat <- na.omit(results_low_sleepers[results_low_sleepers$n_events > 20, ])
dat <- dat[-which(dat$concept_name == "Morbid obesity")]
dat$label <- NA
dat$label[dat$p_value < .001] <- dat$concept_name[dat$p_value < .001]
dat$label <- do.call(rbind,strsplit(dat$label,","))[,1]
dat$siglevel_cut <- cut(dat$p_value,breaks = c(0,.00001,.0001,.001),right=FALSE,labels = FALSE)
dat$siglevel[dat$siglevel_cut == 1] <- "p < .00001"
dat$siglevel[dat$siglevel_cut == 2] <- "p < .0001"
dat$siglevel[dat$siglevel_cut == 3] <- "p < .001"
dat$siglevel = as.factor(dat$siglevel)
dat$siglevel = factor(x=dat$siglevel,levels=c("p < .00001","p < .0001", "p < .001"))

# Convert odds_ratio to numeric BEFORE filtering
dat$odds_ratio <- as.numeric(dat$odds_ratio)

# Remove outlier results (all non-significant) 
dat_plot <- dat[dat$odds_ratio < 5 & !is.na(dat$odds_ratio),]

options(repr.plot.res=200,repr.plot.width=10,repr.plot.height=10)
ggplot(data=dat_plot, aes(x=odds_ratio, y=-log10(p_value), color=siglevel, label=label)) +
  geom_point(color='grey60') + 
  theme_bw() +
  scale_x_continuous(breaks = seq(0,5,by=.25)) +
  scale_y_continuous(breaks = seq(0,10,by=2)) +
  geom_vline(xintercept = 1) +
  scale_color_viridis(discrete=TRUE,
                      na.translate=FALSE,
                      name="Sig. Level",
                      begin = .1,end=.8) +
  geom_text_repel(max.overlaps = 25,
                  box.padding = .5,
                  force = 1.70,
                  max.iter = 500000,
                  max.time = 21) +
  ylab(bquote(~-log[10]~ 'p-value')) +
  xlab("Odds Ratio") +
  geom_hline(yintercept=-log10(.05/nrow(dat)), col="red") +
  theme(panel.grid.minor = element_blank(),
        panel.border = element_rect(),
        text = element_text(size=18),
        axis.title = element_text(size=20),
        axis.text = element_text(size=10))+
  ggtitle("Low Sleep Stratum PGRS")


In [ ]:
df_plot <-results_low_sleepers

# Create the -log10(p-value) column
df_plot$minus_log10_p <- -log10(df_plot$p_value)

# Handle zero p-values (replace with very small number to avoid infinite values)
df_plot$minus_log10_p[df_plot$p_value == 0] <- 5  # or use a large number


# Add color coding based on significance and effect direction
df_plot$significance <- ifelse(df_plot$p_value < 0.05/nrow(df_plot), "Bonferroni Significant",
                              ifelse(df_plot$p_value < 0.05, "Nominally Significant", "Not Significant"))

df_plot$effect_direction <- ifelse(df_plot$estimate > 0, "Positive", "Negative")

# Create color variable combining significance and direction
df_plot$color_group <- paste(df_plot$significance, df_plot$effect_direction, sep = " - ")

ggplot(df_plot, aes(x = phecode, y = minus_log10_p, color = color_group)) +
  geom_point(alpha = 0.7, size = 2) +
  geom_hline(yintercept = -log10(0.05), color = "red", linetype = "dashed", alpha = 0.7) +
  geom_hline(yintercept = -log10(0.05/nrow(df_plot)), color = "blue", linetype = "dashed", alpha = 0.7) +
  scale_color_manual(values = c(
    "Bonferroni Significant - Positive" = "darkred",
    "Bonferroni Significant - Negative" = "darkblue", 
    "Nominally Significant - Positive" = "red",
    "Nominally Significant - Negative" = "blue",
    "Not Significant - Positive" = "lightcoral",
    "Not Significant - Negative" = "lightblue"
  )) +
  labs(title = "PheWAS Manhattan Plot - Sleep Deviation Effects",
       x = "Phecode", 
       y = "-log10(p-value)",
       color = "Significance & Direction",
       subtitle = paste("Red line: p=0.05, Blue line: Bonferroni p=", 
                       round(0.05/nrow(df_plot), 6))) +
  theme_minimal() +
  theme(axis.text.x = element_text(angle = 45, hjust = 1),
        legend.position = "bottom")


In [ ]:
head(results_low_sleepers[order(results_low_sleepers$p_value,decreasing=F),])

## Interaction Test

In [ ]:
#Extract concordant_data
all_sleepers_data <- final_dat
all_sleepers_data$sleep_z <- scale(all_sleepers_data$average_daily_sleep)

In [ ]:
required_vars <- c("score_z", "sleep_z", "age", "sex_concept", 
                   paste0("pca_", 1:5))
missing_vars <- setdiff(required_vars, colnames(all_sleepers_data))
print(missing_vars)


In [ ]:
test_phe <- "has_phe_8"  # One of the failing phenotypes
test_data <- all_sleepers_data[!is.na(all_sleepers_data[[test_phe]]), ]
test_data[[test_phe]] <- test_data[[test_phe]] >= 1

# Try fitting manually
test_formula <- as.formula(paste(test_phe, "~ score_z * sleep_z + age + I(age^2) + sex_concept + pca_1 + pca_2 + pca_3 + pca_4 + pca_5"))
test_model <- glm(test_formula, data = test_data, family = binomial())


In [ ]:
run <- TRUE
if(run){

results_all_interaction_sleepers <- run_phewas(all_sleepers_data,
                     model_equation = "score_z * sleep_z + age + I(age^2) + sex_concept+pca_1+pca_2+pca_3+pca_4+pca_5",
                     exposure_var = "score_z:sleep_z",
                     impute = FALSE,
                     save_file_path = "./PGRS_Average/results_all_interaction_sleepers.csv",
                     save_models = TRUE,
                     model_save_path = "./PGRS_Average/results_all_interaction_sleepers_models/",
                     phemap = phemap)
    }else{
    results_all_interaction_sleepers <- data.table::fread("./PGRS_Average/results_all_interaction_sleepers.csv")
}

In [ ]:
dat <- na.omit(results_all_interaction_sleepers)
# kept it in. Need to look into why it was removed. 
dat <- dat[-which(dat$concept_name == "Morbid obesity")] #Removed this in publication and so doing same here
dat$label <- NA
dat$label[dat$p_value < .001] <- dat$concept_name[dat$p_value < .001]
dat$label <- do.call(rbind,strsplit(dat$label,","))[,1]
dat$siglevel_cut <- cut(dat$p_value,breaks = c(0,.00001,.0001,.001),right=FALSE,labels = FALSE)
dat$siglevel[dat$siglevel_cut == 1] <- "p < .00001"
dat$siglevel[dat$siglevel_cut == 2] <- "p < .0001"
dat$siglevel[dat$siglevel_cut == 3] <- "p < .001"
dat$siglevel = as.factor(dat$siglevel)
dat$siglevel = factor(x=dat$siglevel,levels=c("p < .00001","p < .0001", "p < .001"))
# Remove outlier results (all non-significant) 
dat_plot <- dat[dat$odds_ratio < 5,]



options(repr.plot.res=200,repr.plot.width=10,repr.plot.height=10)
ggplot(data=dat_plot, aes(x=odds_ratio, y=-log10(p_value), color=siglevel, label=label)) +
  geom_point(color='grey60') + 
  theme_bw() +
  scale_x_continuous(breaks = seq(0,5,by=.25)) +
  scale_y_continuous(breaks = seq(0,10,by=2)) +
  geom_vline(xintercept = 1) +
  scale_color_viridis(discrete=TRUE,
                      na.translate=FALSE,
                      name="Sig. Level",
                      begin = .1,end=.8) +
  geom_text_repel(max.overlaps = 25,
                  box.padding = .5,
                  force = 1.70,
                  max.iter = 500000,
                  max.time = 21) +
  ylab(bquote(~-log[10]~ 'p-value')) +
  xlab("Odds Ratio") +
  geom_hline(yintercept=-log10(.05/nrow(dat)), col="red") +
  theme(panel.grid.minor = element_blank(),
        panel.border = element_rect(),
        text = element_text(size=18),
        axis.title = element_text(size=20),
        axis.text = element_text(size=10))
ggtitle("Concordant Sleep PGRS Abs")



In [ ]:
df_plot <-results_all_interaction_sleepers

# Create the -log10(p-value) column
df_plot$minus_log10_p <- -log10(df_plot$p_value)

# Handle zero p-values (replace with very small number to avoid infinite values)
df_plot$minus_log10_p[df_plot$p_value == 0] <- 5  # or use a large number


# Add color coding based on significance and effect direction
df_plot$significance <- ifelse(df_plot$p_value < 0.05/nrow(df_plot), "Bonferroni Significant",
                              ifelse(df_plot$p_value < 0.05, "Nominally Significant", "Not Significant"))

df_plot$effect_direction <- ifelse(df_plot$estimate > 0, "Positive", "Negative")

# Create color variable combining significance and direction
df_plot$color_group <- paste(df_plot$significance, df_plot$effect_direction, sep = " - ")

ggplot(df_plot, aes(x = phecode, y = minus_log10_p, color = color_group)) +
  geom_point(alpha = 0.7, size = 2) +
  geom_hline(yintercept = -log10(0.05), color = "red", linetype = "dashed", alpha = 0.7) +
  geom_hline(yintercept = -log10(0.05/nrow(df_plot)), color = "blue", linetype = "dashed", alpha = 0.7) +
  scale_color_manual(values = c(
    "Bonferroni Significant - Positive" = "darkred",
    "Bonferroni Significant - Negative" = "darkblue", 
    "Nominally Significant - Positive" = "red",
    "Nominally Significant - Negative" = "blue",
    "Not Significant - Positive" = "lightcoral",
    "Not Significant - Negative" = "lightblue"
  )) +
  labs(title = "PheWAS Manhattan Plot - Sleep Deviation Effects",
       x = "Phecode", 
       y = "-log10(p-value)",
       color = "Significance & Direction",
       subtitle = paste("Red line: p=0.05, Blue line: Bonferroni p=", 
                       round(0.05/nrow(df_plot), 6))) +
  theme_minimal() +
  theme(axis.text.x = element_text(angle = 45, hjust = 1),
        legend.position = "bottom")


In [ ]:
head(results_all_interaction_sleepers[order(results_all_interaction_sleepers$p_value,decreasing=F),])

## Mismatch Quanitfy

In [ ]:
all_sleepers_data <- final_dat %>%
  mutate(mismatch = - sleep.dev * sign(score_z)) 
  # mismatch>0: you’re sleeping opposite to your genetic tendency

In [ ]:
run <- TRUE
if(run){
    results_all_mismatch <- run_phewas(all_sleepers_data,
                         model_equation = "mismatch + age + I(age^2) + sex_concept+pca_1+pca_2+pca_3+pca_4+pca_5",
                         exposure_var = "mismatch",
                         impute = TRUE,
                         save_file_path = "./PGRS_Average/results_all_mismatch.csv",
                         save_models = TRUE,
                         model_save_path = "./PGRS_Average/results_all_mismatchs_models/",
                         phemap = phemap)
        }else{
        results_all_interaction_sleepers <- data.table::fread("./PGRS_Average/results_all_mismatch.csv")
}

In [ ]:
dat <- na.omit(results_all_mismatch)
# kept it in. Need to look into why it was removed. 
dat <- dat[-which(dat$concept_name == "Morbid obesity")] #Removed this in publication and so doing same here
dat$label <- NA
dat$label[dat$p_value < .001] <- dat$concept_name[dat$p_value < .001]
dat$label <- do.call(rbind,strsplit(dat$label,","))[,1]
dat$siglevel_cut <- cut(dat$p_value,breaks = c(0,.00001,.0001,.001),right=FALSE,labels = FALSE)
dat$siglevel[dat$siglevel_cut == 1] <- "p < .00001"
dat$siglevel[dat$siglevel_cut == 2] <- "p < .0001"
dat$siglevel[dat$siglevel_cut == 3] <- "p < .001"
dat$siglevel = as.factor(dat$siglevel)
dat$siglevel = factor(x=dat$siglevel,levels=c("p < .00001","p < .0001", "p < .001"))
# Remove outlier results (all non-significant) 
dat_plot <- dat[dat$odds_ratio < 5,]



options(repr.plot.res=200,repr.plot.width=10,repr.plot.height=10)
ggplot(data=dat_plot, aes(x=odds_ratio, y=-log10(p_value), color=siglevel, label=label)) +
  geom_point(color='grey60') + 
  theme_bw() +
  scale_x_continuous(breaks = seq(0,5,by=.25)) +
  scale_y_continuous(breaks = seq(0,10,by=2)) +
  geom_vline(xintercept = 1) +
  scale_color_viridis(discrete=TRUE,
                      na.translate=FALSE,
                      name="Sig. Level",
                      begin = .1,end=.8) +
  geom_text_repel(max.overlaps = 25,
                  box.padding = .5,
                  force = 1.70,
                  max.iter = 500000,
                  max.time = 21) +
  ylab(bquote(~-log[10]~ 'p-value')) +
  xlab("Odds Ratio") +
  geom_hline(yintercept=-log10(.05/nrow(dat)), col="red") +
  theme(panel.grid.minor = element_blank(),
        panel.border = element_rect(),
        text = element_text(size=18),
        axis.title = element_text(size=20),
        axis.text = element_text(size=10))
ggtitle("Concordant Sleep PGRS Abs")



In [ ]:
df_plot <-results_all_mismatch

# Create the -log10(p-value) column
df_plot$minus_log10_p <- -log10(df_plot$p_value)

# Handle zero p-values (replace with very small number to avoid infinite values)
df_plot$minus_log10_p[df_plot$p_value == 0] <- 5  # or use a large number


# Add color coding based on significance and effect direction
df_plot$significance <- ifelse(df_plot$p_value < 0.05/nrow(df_plot), "Bonferroni Significant",
                              ifelse(df_plot$p_value < 0.05, "Nominally Significant", "Not Significant"))

df_plot$effect_direction <- ifelse(df_plot$estimate > 0, "Positive", "Negative")

# Create color variable combining significance and direction
df_plot$color_group <- paste(df_plot$significance, df_plot$effect_direction, sep = " - ")

ggplot(df_plot, aes(x = phecode, y = minus_log10_p, color = color_group)) +
  geom_point(alpha = 0.7, size = 2) +
  geom_hline(yintercept = -log10(0.05), color = "red", linetype = "dashed", alpha = 0.7) +
  geom_hline(yintercept = -log10(0.05/nrow(df_plot)), color = "blue", linetype = "dashed", alpha = 0.7) +
  scale_color_manual(values = c(
    "Bonferroni Significant - Positive" = "darkred",
    "Bonferroni Significant - Negative" = "darkblue", 
    "Nominally Significant - Positive" = "red",
    "Nominally Significant - Negative" = "blue",
    "Not Significant - Positive" = "lightcoral",
    "Not Significant - Negative" = "lightblue"
  )) +
  labs(title = "PheWAS Manhattan Plot - Sleep Deviation Effects",
       x = "Phecode", 
       y = "-log10(p-value)",
       color = "Significance & Direction",
       subtitle = paste("Red line: p=0.05, Blue line: Bonferroni p=", 
                       round(0.05/nrow(df_plot), 6))) +
  theme_minimal() +
  theme(axis.text.x = element_text(angle = 45, hjust = 1),
        legend.position = "bottom")


In [ ]:
head(results_all_interaction_sleepers[order(results_all_interaction_sleepers$p_value,decreasing=F),])